# 02 - Data Cleaning (Part 1)
**Goal:** Select relevant columns, keep only rows usable for salary analysis, remove duplicates, and inspect missing values. Outlier handling comes in the next notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

RAW_PATH = Path("../data/raw/results.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.exists(), f"File not found: {RAW_PATH.resolve()}"

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Raw shape: {df_raw.shape}")

Raw shape: (49191, 172)


In [2]:
# Columns we plan to use in the analysis and the model
KEEP_COLS = [
    "ResponseId",
    "ConvertedCompYearly",   # target: annual salary in USD (pre-converted by Stack Overflow)
    "CompTotal",             # original salary as reported (in the respondent's currency)
    "Currency",
    "Country",
    "WorkExp",               # years of professional experience
    "DevType",
    "EdLevel",
    "Employment",
    "RemoteWork",
    "OrgSize",
    "Age",
    "LanguageHaveWorkedWith",
]

missing_cols = [c for c in KEEP_COLS if c not in df_raw.columns]
print("Columns not found in this survey year:", missing_cols or "none")

KEEP_COLS = [c for c in KEEP_COLS if c in df_raw.columns]
df = df_raw[KEEP_COLS].copy()
print(f"Shape after column selection: {df.shape}")

Columns not found in this survey year: none
Shape after column selection: (49191, 13)


In [3]:
dup_ids = df["ResponseId"].duplicated().sum() if "ResponseId" in df.columns else "n/a"
dup_rows = df.duplicated().sum()

print(f"Duplicated ResponseId: {dup_ids}")
print(f"Fully duplicated rows: {dup_rows}")

df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

Duplicated ResponseId: 0
Fully duplicated rows: 0
Shape after dropping duplicates: (49191, 13)


In [4]:
n_before = len(df)

# A salary is usable only if it is present and strictly positive
df = df[df["ConvertedCompYearly"].notna() & (df["ConvertedCompYearly"] > 0)].copy()

print(f"Rows before: {n_before:,}")
print(f"Rows after : {len(df):,}")
print(f"Dropped    : {n_before - len(df):,} ({(n_before - len(df)) / n_before:.1%})")

Rows before: 49,191
Rows after : 23,947
Dropped    : 25,244 (51.3%)


In [5]:
# Look at the real values first, never filter blindly
df["Employment"].value_counts(dropna=False).head(15)

Employment
Employed                                                19500
Independent contractor, freelancer, or self-employed     3050
Student                                                   703
Not employed                                              504
Retired                                                   133
I prefer not to say                                        57
Name: count, dtype: int64

In [6]:
# In some survey years "Employment" is multi-select (values separated by ';').
# We keep anyone who is employed and exclude students or job seekers without a job.
emp = df["Employment"].fillna("")

is_employed = (
    emp.str.contains("Employed", case=False)
    & ~emp.str.contains("Not employed", case=False)
    & ~emp.str.contains("part-time", case=False)
)

print(f"Rows before: {len(df):,}")
df_emp = df[is_employed].copy()
print(f"Rows after : {len(df_emp):,}")

print("\nEmployment values kept:")
print(df_emp["Employment"].value_counts().head(10))

Rows before: 23,947
Rows after : 22,550

Employment values kept:
Employment
Employed                                                19500
Independent contractor, freelancer, or self-employed     3050
Name: count, dtype: int64


In [7]:
# WorkExp should be numeric; convert safely (bad values become NaN)
df_emp["WorkExp"] = pd.to_numeric(df_emp["WorkExp"], errors="coerce")

print(df_emp["WorkExp"].describe())

# Sanity check: professional experience above 50 years is suspicious
print(f"\nRows with WorkExp > 50: {(df_emp['WorkExp'] > 50).sum()}")

count    22308.000000
mean        13.777748
std          9.842777
min          1.000000
25%          6.000000
50%         11.000000
75%         20.000000
max        100.000000
Name: WorkExp, dtype: float64

Rows with WorkExp > 50: 32


In [8]:
missing_report = (
    df_emp.isna().mean().mul(100).round(1)
    .sort_values(ascending=False)
    .to_frame("missing_%")
)
missing_report

,missing_%
RemoteWork,8.7
OrgSize,8.3
LanguageHaveWorkedWith,7.3
WorkExp,1.1
EdLevel,0.1
Country,0.0
Currency,0.0
CompTotal,0.0
ConvertedCompYearly,0.0
ResponseId,0.0


In [9]:
# Strip whitespace from all text columns to avoid duplicate categories
text_cols = df_emp.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df_emp[c] = df_emp[c].str.strip()

# Shorten the very long country names for nicer charts later
df_emp["Country"] = df_emp["Country"].replace({
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "United States of America": "United States",
})

df_emp["Country"].value_counts().head(10)

Country
United States     4932
Germany           1973
United Kingdom    1418
India             1015
France             967
Canada             865
Ukraine            681
Brazil             603
Netherlands        589
Poland             563
Name: count, dtype: int64

In [10]:
OUT_PATH = PROCESSED_DIR / "stage1_salary_rows.csv"
df_emp.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH.resolve()}")
print(f"Final shape for this stage: {df_emp.shape}")

Saved: D:\tech-salary-analysis\data\processed\stage1_salary_rows.csv
Final shape for this stage: (22550, 13)
